# Evolutionary Stability in a Concave Public Goods Game
## Three Preference Types: Homo Oeconomicus, Homo Kantiensis, Pure Altruist

| Label | Type | Utility | Equilibrium Strategy |
|---|---|---|---|
| H | Homo Oeconomicus | $u_H = \pi(x,y)$ | $x_H^* = \frac{r-1}{2r}$ |
| K | Homo Kantiensis | $u_K = \pi(x,x)$ | $x_K^* = \frac{2r-1}{4r}$ (= social optimum) |
| A | Pure Altruist | $u_A = \pi(y,x)$ | $x_A^* = \frac{1}{2}$ |

**Key ordering:** $x_H^* < x_K^* < x_A^*$ for all $r > 1$.  
Homo Kantiensis coincides with the social optimum.  
Pure Altruist over-contributes beyond the social optimum.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'axes.titlesize': 13,
                     'figure.dpi': 120})

## 1. Game Setup

**Concave Public Goods Game:**
$$\pi(x_i, x_j) = r(x_i + x_j) - \frac{r(x_i+x_j)^2}{2} - x_i, \quad r > 1$$

Marginal return to total contribution is **decreasing** (congestion effect).  
Social dilemma: each player has a free-riding incentive.

**Equilibrium strategies:**

| Type | Utility maximised | FOC | $x^*$ |
|---|---|---|---|
| H (Homo Oec.) | $\pi(x,y)$ | $r - r(x+y) - 1 = 0$ | $x_H^* = \frac{r-1}{2r}$ |
| K (Homo Kant.) | $\pi(x,x)$ | $(2r-1) - 4rx = 0$ | $x_K^* = \frac{2r-1}{4r} = x_{\text{soc}}^*$ |
| A (Altruist) | $\pi(y,x)$ | $r(1-x_j-x_i) = 0$ | $x_A^* = \frac{1}{2}$ |

In [ ]:
r = 3.0

def pi(x, y):
    z = x + y
    return r * z - r * z**2 / 2 - x

x_H = (r - 1) / (2 * r)           # Homo Oeconomicus
x_K = (2 * r - 1) / (4 * r)       # Homo Kantiensis = social optimum
x_A = 0.5                          # Pure Altruist
x_soc = x_K

STRATEGIES = np.array([x_H, x_K, x_A])
NAMES  = ['Homo Oeconomicus (H)', 'Homo Kantiensis (K)', 'Pure Altruist (A)']
LABELS = ['H', 'K', 'A']
COLORS = ['#e74c3c', '#f39c12', '#3498db']

print(f'r = {r}')
print(f'  H (Homo Oec.)   x_H* = {x_H:.4f}  (partial free-rider)')
print(f'  K (Homo Kant.)  x_K* = {x_K:.4f}  (= social optimum)')
print(f'  A (Altruist)    x_A* = {x_A:.4f}  (over-contributor)')
print()
print('Own-type payoffs:')
for name, x in zip(NAMES, STRATEGIES):
    print(f'  {name}: pi(x*,x*) = {pi(x,x):.4f}')

In [ ]:
xgrid = np.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left: each type's objective function, opponent fixed at that type's equilibrium
ax = axes[0]
ax.plot(xgrid, [pi(x, x_H) for x in xgrid],
        color=COLORS[0], lw=1.8,
        label=f'H objective: π(x, {x_H:.3f})')
ax.plot(xgrid, [pi(x, x) for x in xgrid],
        color=COLORS[1], lw=1.8, ls='--',
        label='K objective: π(x, x)')
ax.plot(xgrid, [pi(x_A, x) for x in xgrid],
        color=COLORS[2], lw=1.8, ls=':',
        label=f'A objective: π({x_A:.2f}, x)')

for x, col, lbl in zip(STRATEGIES, COLORS, LABELS):
    ax.axvline(x, color=col, lw=1.2, alpha=0.6)
    ax.scatter([x], [pi(x, x)], color=col, s=70, zorder=5,
               label=f'{lbl}: x*={x:.3f}')
ax.axvline(x_soc, color='black', lw=1, ls='-.', label=f'Social opt. {x_soc:.3f}')
ax.set_xlabel('Strategy x')
ax.set_ylabel('Objective value')
ax.set_title('Objective functions and equilibria\n(opponent plays own equilibrium)')
ax.legend(fontsize=7)

# Right: material payoff as opponent's strategy varies
ax = axes[1]
for x, col, name in zip(STRATEGIES, COLORS, NAMES):
    ax.plot(xgrid, [pi(x, y) for y in xgrid], color=col, lw=2,
            label=f'{name} (x*={x:.3f})')
ax.set_xlabel("Opponent's strategy y")
ax.set_ylabel('Material payoff π(x*, y)')
ax.set_title('Material payoff vs. opponent strategy')
ax.legend(fontsize=9)

plt.suptitle(f'Concave Public Goods Game (r={r})', fontsize=13)
plt.tight_layout()
plt.savefig('pgg_strategies.png', dpi=150)
plt.show()

## 2. Fitness and Replicator Dynamics

With assortativity index $\sigma \in [0,1]$, the fitness of type $i$ is:
$$f_i = \sigma\,\pi(x_i^*, x_i^*) + (1-\sigma)\sum_j s_j\,\pi(x_i^*, x_j^*)$$

Standard replicator dynamics:
$$s_i(t+1) = s_i(t) + s_i(t)\,(f_i(t) - \bar{f}(t)), \quad \bar{f} = \sum_j s_j f_j$$

Fitness measures **material payoff** $\pi$, not each type's own utility function.

In [ ]:
def fitness(s, sigma):
    f_self  = np.array([pi(x_H, x_H), pi(x_K, x_K), pi(x_A, x_A)])
    f_cross = np.array([
        s[0]*pi(x_H, x_H) + s[1]*pi(x_H, x_K) + s[2]*pi(x_H, x_A),
        s[0]*pi(x_K, x_H) + s[1]*pi(x_K, x_K) + s[2]*pi(x_K, x_A),
        s[0]*pi(x_A, x_H) + s[1]*pi(x_A, x_K) + s[2]*pi(x_A, x_A),
    ])
    return sigma * f_self + (1.0 - sigma) * f_cross


def simulate(s0, sigma, n_steps):
    s = np.array(s0, dtype=float)
    history = np.zeros((n_steps + 1, 3))
    history[0] = s
    for t in range(n_steps):
        f     = fitness(s, sigma)
        f_bar = s @ f
        s     = s + s * (f - f_bar)
        s     = np.clip(s, 0, 1)
        s    /= s.sum()
        history[t + 1] = s
    return history


s_equal = np.array([1/3, 1/3, 1/3])
print('Fitness at equal shares (sigma=0):')
for name, fi in zip(NAMES, fitness(s_equal, 0.0)):
    print(f'  {name}: f = {fi:.4f}')
print()
print('Fitness at equal shares (sigma=1):')
for name, fi in zip(NAMES, fitness(s_equal, 1.0)):
    print(f'  {name}: f = {fi:.4f}')

In [ ]:
N_STEPS      = 800
S0           = [1/3, 1/3, 1/3]
SIGMA_VALS   = [0.0, 0.25, 0.40, 0.80]
SIGMA_COLORS = ['#c0392b', '#e67e22', '#27ae60', '#2980b9']

results = {sigma: simulate(S0, sigma, N_STEPS) for sigma in SIGMA_VALS}

print('Final shares:')
for sigma in SIGMA_VALS:
    f = results[sigma][-1]
    print(f'  sigma={sigma}: H={f[0]:.3f}  K={f[1]:.3f}  A={f[2]:.3f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
for ax, sigma in zip(axes.flatten(), SIGMA_VALS):
    hs = results[sigma]
    t  = np.arange(N_STEPS + 1)
    ax.plot(t, hs[:, 0], color=COLORS[0], lw=2.2, label='Homo Oeconomicus (H)')
    ax.plot(t, hs[:, 1], color=COLORS[1], lw=2.2, label='Homo Kantiensis (K)')
    ax.plot(t, hs[:, 2], color=COLORS[2], lw=2.2, ls='--', label='Pure Altruist (A)')
    ax.set_title(f'σ = {sigma}')
    ax.set_xlabel('Generation')
    ax.set_ylabel('Population share')
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=9)
fig.suptitle(
    f'Replicator Dynamics — Concave Public Goods Game (r={r})',
    fontsize=13
)
plt.tight_layout()
plt.savefig('pgg_population.png', dpi=150)
plt.show()

In [ ]:
sigma_grid   = np.linspace(0, 1, 120)
final_shares = np.zeros((len(sigma_grid), 3))
for i, sigma in enumerate(sigma_grid):
    final_shares[i] = simulate(S0, sigma, N_STEPS)[-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.stackplot(sigma_grid,
             final_shares[:, 0], final_shares[:, 1], final_shares[:, 2],
             labels=['Homo Oeconomicus (H)', 'Homo Kantiensis (K)', 'Pure Altruist (A)'],
             colors=COLORS, alpha=0.82)
ax.set_xlabel('Assortativity index σ')
ax.set_ylabel('Steady-state share')
ax.set_title(f'Phase Diagram: Steady-State Composition vs. σ  (r={r})')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.legend(loc='upper center', fontsize=9)
plt.tight_layout()
plt.savefig('pgg_phase.png', dpi=150)
plt.show()

In [ ]:
def to_cart(sH, sK, sA):
    return sK + 0.5*sA, (np.sqrt(3)/2)*sA

def draw_simplex(ax, sigma):
    vx = [0, 1, 0.5, 0]; vy = [0, 0, np.sqrt(3)/2, 0]
    ax.plot(vx, vy, 'k-', lw=1.5)
    ax.text(-0.14, -0.14, f'H\nx*={x_H:.2f}',
            ha='center', va='top', fontsize=8, color=COLORS[0])
    ax.text( 1.14, -0.14, f'K\nx*={x_K:.2f}',
            ha='center', va='top', fontsize=8, color=COLORS[1])
    ax.text(  0.5,  np.sqrt(3)/2 + 0.10, f'A\nx*={x_A:.2f}',
            ha='center', va='bottom', fontsize=8, color=COLORS[2])
    ax.text(  0.5, 0.06, f'σ = {sigma}',
            ha='center', va='bottom', fontsize=11, fontweight='bold', color='#333333')
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-0.28, 1.28)
    ax.set_ylim(-0.28, np.sqrt(3)/2 + 0.22)

starts = [
    [1/3, 1/3, 1/3],
    [0.1, 0.6, 0.3], [0.1, 0.3, 0.6],
    [0.6, 0.2, 0.2], [0.2, 0.6, 0.2], [0.2, 0.2, 0.6],
    [0.5, 0.4, 0.1], [0.5, 0.1, 0.4],
]

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
for ax, sigma, sc in zip(axes, SIGMA_VALS, SIGMA_COLORS):
    draw_simplex(ax, sigma)
    for s0t in starts:
        hs = simulate(s0t, sigma, N_STEPS)
        xs, ys = to_cart(hs[:, 0], hs[:, 1], hs[:, 2])
        ax.plot(xs, ys, color=sc, alpha=0.5, lw=1.2)
        mid = len(xs) // 2
        ax.annotate('', xy=(xs[mid+5], ys[mid+5]), xytext=(xs[mid], ys[mid]),
                    arrowprops=dict(arrowstyle='->', color=sc, lw=1.2))
        ax.scatter(xs[-1], ys[-1], color=sc, s=25, zorder=5)

fig.suptitle(f'Simplex Phase Portrait — Concave Public Goods Game (r={r})', fontsize=13)
plt.tight_layout()
plt.savefig('pgg_simplex.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

f_grid = np.zeros((len(sigma_grid), 3))
for i, sigma in enumerate(sigma_grid):
    f_grid[i] = fitness(np.array([1/3, 1/3, 1/3]), sigma)

ax = axes[0]
for j, (name, col) in enumerate(zip(NAMES, COLORS)):
    ax.plot(sigma_grid, f_grid[:, j], color=col, lw=2, label=name)
ax.set_xlabel('Assortativity index σ')
ax.set_ylabel('Fitness f_i')
ax.set_title('Fitness at equal initial shares')
ax.legend()

ax = axes[1]
x_pos = np.array([0, 1, 2])
bars = ax.bar(x_pos, STRATEGIES, color=COLORS, width=0.5, alpha=0.8)
ax.axhline(x_soc, color='black', ls='--', lw=1.5, label=f'Social optimum {x_soc:.3f}')
ax.set_xticks(x_pos)
ax.set_xticklabels(['Homo Oec.\n(H)', 'Homo Kant.\n(K)', 'Pure Altruist\n(A)'])
ax.set_ylabel('Equilibrium contribution x*')
ax.set_title('Equilibrium contributions')
ax.set_ylim(0, 0.7)
ax.legend()
for bar, x in zip(bars, STRATEGIES):
    ax.text(bar.get_x() + bar.get_width()/2, x + 0.01, f'{x:.3f}',
            ha='center', va='bottom', fontsize=10)

plt.suptitle(f'Strategy and Fitness Analysis — Concave Public Goods Game (r={r})', fontsize=13)
plt.tight_layout()
plt.savefig('pgg_fitness.png', dpi=150)
plt.show()

## Summary

| Regime | Dominant type | Mechanism |
|---|---|---|
| Low σ | Homo Oeconomicus (H) | Free-riding pays under random matching |
| High σ | Homo Kantiensis (K) | Assortative matching rewards Kantian optimum |
| Any σ | A ≺ K always | Over-contribution beyond social optimum is always dominated |

**Why K dominates A at all σ:**  
K plays $x_K^* = x_{\text{soc}}^*$, which maximises total welfare under concave returns.  
A's extra contribution beyond $x_K^*$ produces less public good than it costs — A is strictly dominated by K.

**Result aligns with Alger & Weibull (2013):** the evolutionarily stable preference type is homo moralis with Kantian weight $\kappa^* = \sigma$.